# Pandas Tutorial

A hands-on, runnable reference for the pandas data-analysis library. Run each code cell as you go — feel free to tweak values and re-run.


In [ ]:
import pandas as pd
import numpy as np

print(pd.__version__)


## 1. Series: A 1D Labeled Array

- A `Series` is a one-dimensional array with an attached **index** (labels for each value).
- Think of it as a single column of a spreadsheet, or a Python `dict` with ordering and vectorized operations.


In [ ]:
# From a list -> default integer index
s = pd.Series([10, 20, 30, 40])
print(s)

# From a list with a custom index
s2 = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s2)

# From a dict -> keys become the index
s3 = pd.Series({"apple": 5, "banana": 3, "cherry": 8})
print(s3)

print(s2["b"])     # label-based access
print(s2 * 2)       # vectorized operations, like NumPy


## 2. DataFrame: A 2D Labeled Table

- A `DataFrame` is a 2D table: rows + columns, each column effectively a `Series`.
- Common ways to build one: from a dict of lists, a list of dicts, or a NumPy array.


In [ ]:
# From a dict of lists (keys -> column names)
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "age": [25, 30, 35, 28],
    "city": ["NYC", "LA", "NYC", "SF"],
    "salary": [70000, 85000, 62000, 95000],
})
print(df)

# From a list of dicts (each dict -> one row)
df2 = pd.DataFrame([
    {"x": 1, "y": 2},
    {"x": 3, "y": 4},
])
print(df2)


## 3. Inspecting a DataFrame

- `.head()`/`.tail()` — preview rows.
- `.shape`, `.columns`, `.index`, `.dtypes` — structure.
- `.info()` — column types and non-null counts.
- `.describe()` — summary statistics for numeric columns.


In [ ]:
print(df.head(2))     # first 2 rows
print(df.shape)        # (rows, columns)
print(df.columns.tolist())
print(df.dtypes)
df.info()
print(df.describe())   # count, mean, std, min, quartiles, max


## 4. Selecting Data: Columns, Rows, `loc`, and `iloc`

- `df['col']` — select a single column (returns a `Series`); `df[['col1', 'col2']]` — select multiple columns (returns a `DataFrame`).
- `.loc[row_label, col_label]` — select by **label**.
- `.iloc[row_pos, col_pos]` — select by **integer position**.


In [ ]:
print(df["name"])            # single column -> Series
print(df[["name", "age"]])    # multiple columns -> DataFrame

print(df.loc[0])              # row with label/index 0
print(df.loc[0, "name"])       # specific cell by label
print(df.loc[0:2, ["name", "age"]])  # label-based slicing, end IS included

print(df.iloc[0])              # first row by position
print(df.iloc[0, 1])            # first row, second column
print(df.iloc[0:2, 0:2])        # position-based slicing, end NOT included


## 5. Filtering with Boolean Conditions

- Conditions on a column produce a boolean `Series`; pass that inside `df[...]` to keep only matching rows.
- Combine conditions with `&` (and), `|` (or), `~` (not) — each condition must be wrapped in parentheses.


In [ ]:
print(df[df["age"] > 28])

print(df[(df["city"] == "NYC") & (df["age"] > 26)])

print(df[df["city"].isin(["LA", "SF"])])

print(df.query("age > 28 and city == 'NYC'"))   # same filter, SQL-like syntax


## 6. Adding, Modifying, and Dropping Columns

- Assign to a new column label to create it; operations on existing columns are vectorized (no manual loops needed).
- `.drop(columns=[...])` removes columns (returns a new DataFrame by default; use `inplace=True` to modify in place).


In [ ]:
df["bonus"] = df["salary"] * 0.10             # new column from a vectorized expression
df["seniority"] = df["age"].apply(lambda a: "senior" if a >= 30 else "junior")
print(df)

df_dropped = df.drop(columns=["bonus"])         # returns a new DataFrame
print(df_dropped.columns.tolist())
print(df.columns.tolist())                       # original df is untouched


## 7. Handling Missing Data

- Missing values show up as `NaN`. `.isna()`/`.notna()` detect them.
- `.dropna()` removes rows/columns with missing values; `.fillna()` fills them in.


In [ ]:
df_missing = pd.DataFrame({
    "a": [1, np.nan, 3, np.nan],
    "b": [np.nan, 2, 3, 4],
})
print(df_missing)
print(df_missing.isna())

print(df_missing.dropna())                # drops any row containing NaN
print(df_missing.fillna(0))                # fills NaN with 0
print(df_missing.fillna(df_missing.mean()))  # fills NaN with each column's mean


## 8. Sorting

- `.sort_values(by=...)` sorts by column value(s); `.sort_index()` sorts by the index.


In [ ]:
print(df.sort_values(by="age"))
print(df.sort_values(by="age", ascending=False))
print(df.sort_values(by=["city", "age"]))   # sort by multiple columns


## 9. Grouping and Aggregating with `groupby`

- `groupby` implements the classic **split-apply-combine** pattern: split rows into groups by a key, apply an aggregation to each group, then combine the results.


In [ ]:
print(df.groupby("city")["salary"].mean())      # average salary per city

print(df.groupby("city").agg(
    avg_salary=("salary", "mean"),
    max_age=("age", "max"),
    count=("name", "count"),
))

print(df.groupby("city").size())                  # number of rows per group


## 10. Combining DataFrames: `merge` and `concat`

- `pd.merge` joins two DataFrames on a common key, like a SQL `JOIN` (`how='inner'|'left'|'right'|'outer'`).
- `pd.concat` stacks DataFrames on top of each other (rows) or side by side (columns).


In [ ]:
employees = pd.DataFrame({
    "emp_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "dept_id": [10, 20, 10, 30],
})

departments = pd.DataFrame({
    "dept_id": [10, 20, 40],
    "dept_name": ["Engineering", "Sales", "Marketing"],
})

print(pd.merge(employees, departments, on="dept_id", how="inner"))  # only matching dept_ids
print(pd.merge(employees, departments, on="dept_id", how="left"))    # keep all employees

more_employees = pd.DataFrame({"emp_id": [5], "name": ["Eve"], "dept_id": [20]})
print(pd.concat([employees, more_employees], ignore_index=True))       # stack rows


## 11. Transforming Data: `apply`, `map`, and Vectorized String Methods

- `.apply(func)` runs a function across each element/row/column — flexible but slower than built-in vectorized ops.
- `.map(func_or_dict)` on a `Series` transforms each value, or substitutes values using a mapping dict.
- The `.str` accessor gives vectorized string operations (no manual loop needed).


In [ ]:
print(df["name"].map(str.upper))                     # apply a function to every value
print(df["city"].map({"NYC": "New York", "LA": "Los Angeles", "SF": "San Francisco"}))

print(df["name"].str.lower())                          # vectorized string method
print(df["name"].str.contains("a", case=False))

print(df.apply(lambda row: f"{row['name']} ({row['age']})", axis=1))  # apply across rows


## 12. Reading and Writing Files

- `pd.read_csv` / `df.to_csv` — the most common I/O pair. Similar functions exist for Excel (`read_excel`), JSON (`read_json`), SQL (`read_sql`), and more.


In [ ]:
csv_path = "sample_data.csv"
df.to_csv(csv_path, index=False)          # write to disk, without the DataFrame index column

df_from_csv = pd.read_csv(csv_path)
print(df_from_csv)

import os
os.remove(csv_path)                        # clean up the file created for this example


## 13. Pivot Tables

- `pd.pivot_table` reshapes data: choose rows (`index`), columns, and an aggregated value — similar to a spreadsheet pivot table.


In [ ]:
sales = pd.DataFrame({
    "region": ["East", "East", "West", "West", "East", "West"],
    "product": ["A", "B", "A", "B", "A", "A"],
    "revenue": [100, 150, 200, 130, 120, 90],
})

pivot = pd.pivot_table(sales, index="region", columns="product", values="revenue", aggfunc="sum")
print(pivot)


## 14. Working with Dates and Time Series

- `pd.to_datetime` parses strings into `Timestamp` objects.
- A `DatetimeIndex` unlocks date-aware operations: slicing by date range, resampling (`.resample`), and rolling windows (`.rolling`).


In [ ]:
dates = pd.date_range(start="2026-01-01", periods=6, freq="D")
ts = pd.Series([10, 12, 9, 15, 14, 20], index=dates)
print(ts)

print(ts["2026-01-02":"2026-01-04"])   # slice by date range

print(ts.resample("3D").sum())          # bucket into 3-day windows and sum

print(ts.rolling(window=2).mean())      # 2-period moving average


## 15. Multi-Level Indexing (`MultiIndex`)

- A DataFrame can be indexed by more than one key at once, useful for hierarchical/grouped data. `groupby` with multiple keys naturally produces one.


In [ ]:
grouped = df.groupby(["city", "seniority"])["salary"].mean()
print(grouped)

print(grouped.index)              # a MultiIndex: (city, seniority) pairs
print(grouped.loc["NYC"])          # slice the outer level
print(grouped.unstack())            # pivot the inner level into columns


## 16. Summary Cheat Sheet

| Task | Function/Method |
|---|---|
| Create a table | `pd.DataFrame({...})` |
| Inspect | `.head()`, `.info()`, `.describe()` |
| Select | `df['col']`, `.loc[]`, `.iloc[]` |
| Filter | `df[condition]`, `.query()` |
| Clean | `.isna()`, `.dropna()`, `.fillna()` |
| Sort | `.sort_values()`, `.sort_index()` |
| Aggregate | `.groupby().agg()` |
| Combine | `pd.merge()`, `pd.concat()` |
| Transform | `.apply()`, `.map()`, `.str.*` |
| I/O | `pd.read_csv()`, `.to_csv()` |
| Reshape | `pd.pivot_table()`, `.unstack()` |
| Time series | `pd.to_datetime()`, `.resample()`, `.rolling()` |
